In [ ]:
%pip install requests pandas BeautifulSoup openai

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import openai
import json

In [2]:
# Define the URL of the customer stories page
BASE_URL = "https://www.genesys.com/customer-stories"

# Headers to mimic a browser request
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

# Step 1: Fetch the page content using requests
response = requests.get(BASE_URL, headers=HEADERS)
response.raise_for_status()  # Ensure the request was successful

In [3]:
response

<Response [200]>

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

# Step 3: Locate customer story cards with the 'data-tax_products_programs' attribute
filtered_cards = soup.select("div.grid-item[data-tax_products_programs]")

In [5]:
# Step 4: Extract links from filtered cards based on product criteria
story_links = []
for card in filtered_cards:
    # Check if the product matches "Genesys Cloud" or "Genesys Cloud EX"
    product_data = card.get("data-tax_products_programs", "").lower()
    if "genesys cloud" in product_data or "genesys cloud ex" in product_data:
        # Find the <a> tag inside the card and extract the href attribute
        link_tag = card.find("a", href=True)
        if link_tag:
            story_links.append(link_tag["href"])

In [6]:
story_links

['https://www.genesys.com/customer-stories/fanatics/',
 'https://www.genesys.com/customer-stories/prvidr/',
 'https://www.genesys.com/customer-stories/odity/',
 'https://www.genesys.com/customer-stories/grupo-sabin/',
 'https://www.genesys.com/customer-stories/ionos/',
 'https://www.genesys.com/customer-stories/digital-dialog/',
 'https://www.genesys.com/customer-stories/hexaware/',
 'https://www.genesys.com/customer-stories/city-of-clearwater/',
 'https://www.genesys.com/customer-stories/redsalud/',
 'https://www.genesys.com/customer-stories/bac/',
 'https://www.genesys.com/customer-stories/o-phon/',
 'https://www.genesys.com/customer-stories/king-price-insurance/',
 'https://www.genesys.com/customer-stories/pluxee-romania/',
 'https://www.genesys.com/customer-stories/modivcare/',
 'https://www.genesys.com/customer-stories/apm/',
 'https://www.genesys.com/customer-stories/benify/',
 'https://www.genesys.com/customer-stories/fibrus-networks/',
 'https://www.genesys.com/customer-stories

In [7]:
# Replace the openai.api_key with your OpenAI API key
openai.api_key = "OpenAI_API_KEY"

In [25]:
# Function to extract data using OpenAI GPT API
def extract_using_ai(raw_text, fields):
    """
    Extract missing data from unstructured text using OpenAI GPT.

    Args:
        raw_text (str): Raw text content of the page.
        fields (list): List of fields to extract (e.g., "Customer Name", "Industry").

    Returns:
        dict: Extracted fields with their values.
    """
    prompt = f"""
    Extract the following fields from the text below and return the output in JSON format:
    {', '.join(fields)}

    - Do not use extra markdown-like syntax (e.g., the backticks json ... ) around the JSON,
    since these backticks are not valid JSON json.loads fails in python.
    - Only provide the partner if its partner names are specified. It can be name of a company, or organization etc. For example, if the text says "Genesys is partnering with XYZ" or "Partner: XYZ" etc, then XYZ is a partner. Otherwise, leave it blank.
    - Note that XYZ should be a name of company or oganization.
    - Donot include parter as "Ascend to new heights with our partners". Also "Genesys" is not a partner. Instead leave it blank.
    - Industry can be sectors like education, healthcare, finance, cloud, etc.
    - If a field is not found, leave it blank.  For example, if the "Industry" is not found, return "Industry": "".
    
    Text:
    {raw_text}
    """

    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a data extraction assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )

    extracted_data = response["choices"][0]["message"]["content"]
    return extracted_data

In [30]:
# Step 5: Extract additional data from each customer story page
data = []
for link in story_links:
    # Fetch the customer story page
    response = requests.get(link, headers=HEADERS)
    response.raise_for_status()
    # Parse the page content
    page_soup = BeautifulSoup(response.text, "html.parser")

    # Extract Customer Name from end of the link
    customer_name_slug = link.rstrip("/").split("/")[-1]
    customer_name = " ".join(
        word.capitalize() for word in customer_name_slug.split("-")
    )

    # Extract Industry
    industry_tag = page_soup.find("span", string="Industry:")
    industry = (
        industry_tag.find_next_sibling(text=True).strip() if industry_tag else None
    )
    # Extract Location
    location_tag = page_soup.find("span", string="Location:")
    location = (
        location_tag.find_next_sibling(text=True).strip() if location_tag else None
    )

    # Extract Partners
    partners = []
    partner_list = page_soup.select("div.cs-partner-container ul li")
    for partner in partner_list:
        partners.append(partner.get_text(strip=True))

    author = page_soup.select_one("div.quote-speaker p")
    author = author.text.strip() if author else None

    # Format Author
    if author and author.startswith("— "):
        author = author[2:].strip()

    if author and "," in author:  
        author_name, author_designation = author.split(",", 1)
        author_name = author_name.strip()
        author_designation = author_designation.strip()
    else:
        author_name, author_designation = None, None

    # If all fields are None, use OpenAI API as fallback
    if not (
        industry
        or location
        or partners
        or author_name
        or author_designation
    ):
        # Extract raw text content from the page
        raw_text = page_soup.get_text(separator="\n")
        # Use OpenAI to extract missing fields
        missing_fields = [
            "Industry",
            "Location",
            "Partners",
            "Author Name",
            "Author Designation",
        ]
        ai_extracted_data = extract_using_ai(raw_text, missing_fields)

        print(ai_extracted_data)

        try:
            # Parse AI response as JSON
            ai_data = json.loads(ai_extracted_data)
            print("Parsed AI Data:", ai_data)
        except json.JSONDecodeError as e:
            print("Error decoding JSON:", e)

        # Fill the fields from AI extraction
        industry = ai_data.get("Industry", None)
        location = ai_data.get("Location", None)
        partners = ai_data.get("Partners", "").split(", ")
        author_name = ai_data.get("Author Name", None)
        author_designation = ai_data.get("Author Designation", None)

    # Append to data list
    data.append(
        {
            "Customer Name": customer_name,
            "Industry": industry,
            "Location": location,
            "Partners": ", ".join(partners),
            "Author Name": author_name,
            "Author Designation": author_designation,
            "Story Link": link,
        }
    )

E:\Users\nvada\AppData\Local\Temp\ipykernel_324\1177121073.py:21: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  industry_tag.find_next_sibling(text=True).strip() if industry_tag else None
E:\Users\nvada\AppData\Local\Temp\ipykernel_324\1177121073.py:26: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  location_tag.find_next_sibling(text=True).strip() if location_tag else None


{
    "Industry": "Business Process Outsourcing (BPO)",
    "Location": "Cartagena, Colombian Caribbean",
    "Partners": "Global Networks Solutions S.A.",
    "Author Name": "Mauricio Camona",
    "Author Designation": "CEO and Founder"
}
Parsed AI Data: {'Industry': 'Business Process Outsourcing (BPO)', 'Location': 'Cartagena, Colombian Caribbean', 'Partners': 'Global Networks Solutions S.A.', 'Author Name': 'Mauricio Camona', 'Author Designation': 'CEO and Founder'}
{
    "Industry": "Healthcare",
    "Location": "Australia",
    "Partners": "",
    "Author Name": "Melissa Simpson",
    "Author Designation": "Chief Customer Officer"
}
Parsed AI Data: {'Industry': 'Healthcare', 'Location': 'Australia', 'Partners': '', 'Author Name': 'Melissa Simpson', 'Author Designation': 'Chief Customer Officer'}
{
    "Industry": "Travel",
    "Location": "Gold Coast, Australia",
    "Partners": "Flight Centre Travel Group",
    "Author Name": "Conrad Dickson",
    "Author Designation": "General M

In [34]:
data

[{'Customer Name': 'Fanatics',
  'Industry': 'Retail',
  'Location': 'US with global operations',
  'Partners': '',
  'Author Name': 'Nigel Ponds',
  'Author Designation': 'Senior Director, Global Resource Planning and Telephony , Fanatics, Inc.',
  'Story Link': 'https://www.genesys.com/customer-stories/fanatics/'},
 {'Customer Name': 'Prvidr',
  'Industry': 'Software',
  'Location': 'Australia',
  'Partners': '',
  'Author Name': 'Lawrence Drayton',
  'Author Designation': 'Head of Customer Experience, Prvidr',
  'Story Link': 'https://www.genesys.com/customer-stories/prvidr/'},
 {'Customer Name': 'Odity',
  'Industry': 'Customer relations',
  'Location': 'Global',
  'Partners': '',
  'Author Name': 'Clément Polet',
  'Author Designation': 'Deputy CEO Business, Odity',
  'Story Link': 'https://www.genesys.com/customer-stories/odity/'},
 {'Customer Name': 'Grupo Sabin',
  'Industry': 'Healthcare',
  'Location': 'Brazil',
  'Partners': 'InvenIT',
  'Author Name': 'Neilton Alves',
  'Au

In [35]:
# Step 6: Save the extracted data to a CSV file
df = pd.DataFrame(data)
output_file = "customer_stories_data.csv"
df.to_csv(output_file, header=True, index=False)

print(f"Customer stories data saved to {output_file}")

Customer stories data saved to customer_stories_data.csv


In [36]:
df

,Customer Name,Industry,Location,Partners,Author Name,Author Designation,Story Link
0,Fanatics,Retail,US with global operations,,Nigel Ponds,"Senior Director, Global Resource Planning and ...",https://www.genesys.com/customer-stories/fanat...
1,Prvidr,Software,Australia,,Lawrence Drayton,"Head of Customer Experience, Prvidr",https://www.genesys.com/customer-stories/prvidr/
2,Odity,Customer relations,Global,,Clément Polet,"Deputy CEO Business, Odity",https://www.genesys.com/customer-stories/odity/
3,Grupo Sabin,Healthcare,Brazil,InvenIT,Neilton Alves,"Project Analyst, Grupo Sabin",https://www.genesys.com/customer-stories/grupo...
4,Ionos,IT services,Germany with global operations,,Carolin Raezer,"Head of Strategy and Innovation, IONOS",https://www.genesys.com/customer-stories/ionos/
...,...,...,...,...,...,...,...
159,Al Romansiah,Food & Beverage,,,Al Romansiah,Head of IT,https://www.genesys.com/customer-stories/al-ro...
160,Rapid Financial Solutions,Financial Services,,,Daren Jackson,CEO,https://www.genesys.com/customer-stories/rapid...
161,Rose Hulman Institute Of Technology Askrose Ho...,Education,,,Lindsay Hull,AskRose Associate Director for Operations and ...,https://www.genesys.com/customer-stories/rose-...
162,Optimind Winter,Cloud,,,Thibaud Hagerm,Director of Employee Benefits,https://www.genesys.com/customer-stories/optim...
